In [1]:
import pandas as pd
import numpy as np
import os


pi = 3.14159265359

maxval=1e9
minval=1e-9

In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from models.mlp_encoder_model_nonquantized import *

2025-10-07 07:34:50.342688: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-07 07:34:50.344042: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-07 07:34:50.363905: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-07 07:34:50.363922: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-07 07:34:50.363940: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

In [3]:
model=CreateModel_Slim((16,16,2))
model.summary()

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['input_pxls[0][0]']          
 Pooling2D)                                                                                       
                                                                                                  
 average_pooling2d_1 (Avera  (None, 1, 16, 2)             0         ['input_pxls[0][0]']          
 gePooling2D)                                                                                     
                                                                                 

2025-10-07 07:34:51.407423: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [4]:
# get best weights file
pitch = '50x12P5'
batch_size = 5000
fingerprint = '34c2da80'
timeslices = 2
#files = os.listdir('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))
files = os.listdir('weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint, timeslices))

vlosses = [float(f.split("-v")[1].split(".hdf5")[0]) for f in files]
bestfile = files[np.argmin(vlosses)]
#model.load_weights('/data/dajiang/smart-pixels/weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)
model.load_weights('weights/dataset_3src_16x16_weights/weights-{}-bs{}-{}-{}t-mlp_SLIM-manual_input_digitization-checkpoints/'.format(pitch, batch_size, fingerprint, timeslices)+bestfile)

print('Best model: {}'.format(bestfile))

Best model: weights.1883-t39.49-v35.65.hdf5


In [5]:
# load in the test set
test_generator = OptimizedDataGenerator(
    #load_from_tfrecords_dir = '/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/shuffled/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    load_from_tfrecords_dir = '/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/{}t/TFR_val_contained_digitize-manual_mlp-SLIM'.format(timeslices),
    quantize = False # False for soft quantizer and manually quantized inputs
)

Loading metadata from /nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM/metadata.json


In [6]:
# predicts test data
p_test = model.predict(test_generator)

complete_truth = None
for _, y in test_generator:
    if complete_truth is None:
        complete_truth = y
    else:
        complete_truth = np.concatenate((complete_truth, y), axis=0)

# creates df with all predicted values and matrix elements - 4 predictions, all 10 unique matrix elements
df = pd.DataFrame(p_test,columns=['x','y','cotB'])

# stores all true values in same matrix as xtrue, ytrue, etc.
df['xtrue'] = complete_truth[:,0]
df['ytrue'] = complete_truth[:,1]
df['cotBtrue'] = complete_truth[:,2]

# calculates residuals for x, y, cotA, cotB
df['residualsX'] = df['xtrue'] - df['x']
df['residualsY'] = df['ytrue'] - df['y']
df['residualsB'] = df['cotBtrue'] - df['cotB']

# stores results as parquet
#df.to_parquet("/home/dajiang/smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))
df.to_parquet("/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_parquets/TFR_files/{}t-mlp_SLIM-manual_input_digitization-vars.parquet".format(timeslices))

21/21 [==============================] - 1s 32ms/step
